# Sales Lead Conversion Prediction

**Tabular Classification Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `converted`

## 2 · Project Overview

We predict whether a **sales lead will convert** to a paying customer based on their engagement metrics, lead source, and company size.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular classification dataset.
2. Encode categorical features for tree-based models.
3. Build a baseline Logistic Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with accuracy, precision, recall, F1, ROC-AUC, and confusion matrix.

## 4 · Problem Statement

Given a lead's page views, time on site, email opens, demo attendance, lead source, and company size, predict whether they will convert.

## 5 · Why This Project Matters

- **Sales efficiency** — focus reps on high-probability leads.
- Reduces cost per acquisition.
- Directly impacts pipeline velocity and revenue forecasting.
- Lead scoring is a core sales operations tool.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 8,000 |
| **Features** | page_views, time_on_site_min, emails_opened, demos_attended, lead_source, company_size |
| **Target** | converted (0 = not converted, 1 = converted) |
| **Class balance** | ~65/35 |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [1]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


All packages ready.


## 9 · Imports

In [2]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

Imports complete.


## 10 · Configuration / Constants

In [3]:
TARGET = "converted"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

Target: converted
Save dir: E:\Github\Machine-Learning-Projects\Classification\Sales Lead Conversion Prediction


## 11 · Dataset Download or Loading

Synthetic dataset: 8,000 sales leads with engagement metrics and conversion outcome.

In [4]:
np.random.seed(SEED)
n = 8000
page_views = np.random.poisson(5, n)
time_on_site_min = np.round(np.random.exponential(8, n), 1)
emails_opened = np.random.poisson(3, n)
demos_attended = np.random.binomial(3, 0.3, n)
lead_source = np.random.choice(["Organic", "Referral", "Paid", "Social"], n, p=[0.3, 0.25, 0.25, 0.2])
source_num = np.where(lead_source == "Organic", 0, np.where(lead_source == "Referral", 1,
             np.where(lead_source == "Paid", 2, 3)))
company_size = np.random.choice(["Small", "Medium", "Enterprise"], n, p=[0.4, 0.35, 0.25])
size_num = np.where(company_size == "Small", 0, np.where(company_size == "Medium", 1, 2))

score = (0.05 * page_views + 0.03 * time_on_site_min + 0.1 * emails_opened
         + 0.4 * demos_attended + 0.15 * source_num + 0.2 * size_num
         + np.random.normal(0, 0.8, n))
converted = (score > np.percentile(score, 65)).astype(int)

df = pd.DataFrame({"page_views": page_views, "time_on_site_min": time_on_site_min,
                    "emails_opened": emails_opened, "demos_attended": demos_attended,
                    "lead_source": lead_source, "company_size": company_size,
                    "converted": converted})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['converted'].value_counts(normalize=True).round(3)}")
df.head()

Dataset shape: (8000, 7)
Class balance:
converted
0    0.65
1    0.35
Name: proportion, dtype: float64


,page_views,time_on_site_min,emails_opened,demos_attended,lead_source,company_size,converted
0,5,8.8,4,1,Paid,Medium,0
1,4,0.3,5,1,Referral,Small,0
2,4,3.6,6,2,Referral,Medium,1
3,5,2.4,6,2,Organic,Medium,0
4,5,2.0,6,0,Organic,Small,0


## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [5]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed. Value counts:")
print(df[TARGET].value_counts())

DATA VALIDATION

Shape: (8000, 7)

Missing values:
page_views          0
time_on_site_min    0
emails_opened       0
demos_attended      0
lead_source         0
company_size        0
converted           0
dtype: int64

Duplicate rows: 100

Dtypes:
page_views            int32
time_on_site_min    float64
emails_opened         int32
demos_attended        int32
lead_source          object
company_size         object
converted             int64
dtype: object

Target 'converted' confirmed. Value counts:
converted
0    5200
1    2800
Name: count, dtype: int64


## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [6]:
num_cols = df.select_dtypes(include="number").columns.drop(TARGET).tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
for i, col in enumerate(num_cols[:4]):
    ax = axes[i // 2][i % 2]
    df.boxplot(column=col, by=TARGET, ax=ax)
    ax.set_title(col)
plt.suptitle("Feature Distributions by Conversion Status", fontsize=14)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df.groupby("lead_source")[TARGET].mean().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Conversion Rate by Lead Source")
df.groupby("company_size")[TARGET].mean().plot(kind="bar", ax=axes[1], color="steelblue", edgecolor="black")
axes[1].set_title("Conversion Rate by Company Size")
plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `converted`.

In [7]:
fig, ax = plt.subplots(figsize=(6, 4))
df[TARGET].value_counts().plot(kind="bar", ax=ax, color=["salmon", "steelblue"], edgecolor="black")
ax.set_title("Lead Conversion Distribution")
ax.set_xticklabels(["Not Converted (0)", "Converted (1)"], rotation=0)
plt.tight_layout()
plt.show()
print(f"Conversion rate: {df[TARGET].mean():.1%}")

Conversion rate: 35.0%


## 15 · Train/Validation/Test Split Strategy

Stratified 80/20 train/test split preserving class balance.

In [8]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

# Encode target if needed
if y.dtype == object:
    le = LabelEncoder()
    y = pd.Series(le.fit_transform(y), name=TARGET)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Train class distribution:\n{y_train.value_counts(normalize=True).round(3)}")

Train: (6400, 6), Test: (1600, 6)
Train class distribution:
converted
0    0.65
1    0.35
Name: proportion, dtype: float64


## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `lead_source`, `company_size` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Class balance**: ~35% converted — moderate imbalance.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [9]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["engagement_score"] = X_train["page_views"] + X_train["emails_opened"] + 2 * X_train["demos_attended"]
X_test["engagement_score"] = X_test["page_views"] + X_test["emails_opened"] + 2 * X_test["demos_attended"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

Features (7): ['page_views', 'time_on_site_min', 'emails_opened', 'demos_attended', 'lead_source', 'company_size', 'engagement_score']


## 18 · Baseline Model

Logistic Regression as our baseline classifier.

In [10]:
baseline = LogisticRegression(max_iter=1000, random_state=SEED)
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

acc_bl = accuracy_score(y_test, y_pred_bl)
f1_bl = f1_score(y_test, y_pred_bl, average="binary")

print("=" * 50)
print("BASELINE: Logistic Regression")
print("=" * 50)
print(f"Accuracy : {acc_bl:.4f}")
print(f"F1       : {f1_bl:.4f}")
print("\n" + classification_report(y_test, y_pred_bl))

BASELINE: Logistic Regression
Accuracy : 0.7219
F1       : 0.5137

              precision    recall  f1-score   support

           0       0.74      0.88      0.81      1040
           1       0.66      0.42      0.51       560

    accuracy                           0.72      1600
   macro avg       0.70      0.65      0.66      1600
weighted avg       0.71      0.72      0.70      1600



## 19 · LazyPredict Benchmark

Fit dozens of classifiers in one call for a quick ranking.

In [11]:
from lazypredict.Supervised import LazyClassifier

lazy = LazyClassifier(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Classifiers:")
print(lazy_models.head(15).to_string())

LazyPredict — Top 15 Classifiers:
                            Accuracy  Balanced Accuracy   ROC AUC  F1 Score  Precision    Recall  Time Taken
Model                                                                                                       
NearestCentroid             0.678750           0.671703  0.744153  0.684460   0.697653  0.678750    0.018744
Perceptron                  0.682500           0.668819  0.736282  0.686974   0.695324  0.682500    0.026057
LogisticRegression          0.721875           0.652129  0.748950  0.703195   0.712011  0.721875    0.017761
CalibratedClassifierCV      0.721250           0.650824  0.748445  0.702133   0.711364  0.721250    0.046623
LinearDiscriminantAnalysis  0.720625           0.650755  0.748310  0.701861   0.710503  0.720625    0.023671
AdaBoostClassifier          0.716250           0.650275  0.734667  0.699891   0.704932  0.716250    0.160314
LinearSVC                   0.721875           0.650069  0.748440  0.701866   0.712417  0.7218

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [12]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="classification", time_budget=60,
                 metric="f1",
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"Accuracy: {accuracy_score(y_test, y_pred_flaml):.4f}")
print(f"F1: {f1_score(y_test, y_pred_flaml, average='binary'):.4f}")

FLAML best model: lgbm
Accuracy: 0.7081
F1: 0.4918


## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [13]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostClassifier
    t0 = time.perf_counter()
    try:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test).ravel()
    print(f"CatBoost F1: {f1_score(y_test, results['CatBoost'], average='binary'):.4f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM F1: {f1_score(y_test, results['LightGBM'], average='binary'):.4f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBClassifier
    t0 = time.perf_counter()
    try:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  eval_metric="logloss", use_label_encoder=False,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost F1: {f1_score(y_test, results['XGBoost'], average='binary'):.4f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Logistic Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

CatBoost F1: 0.4900  (1.6s)


Training until validation scores don't improve for 30 rounds


Early stopping, best iteration is:
[58]	valid_0's binary_logloss: 0.567099
LightGBM F1: 0.4628  (1.0s)


XGBoost failed: XGBClassifier.fit() got an unexpected keyword argument 'early_stopping_rounds'


## 22 · Top 3 Model Selection

Rank all models by F1 and select the top 3.

In [14]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "Accuracy": round(accuracy_score(y_test, yp), 4),
        "F1": round(f1_score(y_test, yp, average="binary"), 4),
        "Precision": round(precision_score(y_test, yp, average="binary", zero_division=0), 4),
        "Recall": round(recall_score(y_test, yp, average="binary", zero_division=0), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("F1", ascending=False)
print("All Model Rankings (by F1):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

All Model Rankings (by F1):
                     Accuracy      F1  Precision  Recall
Logistic Regression    0.7219  0.5137     0.6620  0.4196
FLAML                  0.7081  0.4918     0.6295  0.4036
CatBoost               0.7125  0.4900     0.6462  0.3946
LightGBM               0.7025  0.4628     0.6288  0.3661

Top 3 models: ['Logistic Regression', 'FLAML', 'CatBoost']


## 23 · Final Training and Evaluation of Top 3

Detailed metrics and confusion matrices.

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    cm = confusion_matrix(y_test, yp)
    ConfusionMatrixDisplay(cm).plot(ax=axes[i], cmap="Blues")
    axes[i].set_title(f"{name}\nF1={f1_score(y_test, yp, average='binary'):.4f}")

plt.suptitle("Top 3 Models — Confusion Matrices", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_confusion_matrices.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    Accuracy : {accuracy_score(y_test, yp):.4f}")
    print(f"    F1       : {f1_score(y_test, yp, average='binary'):.4f}")
    print(f"    Precision: {precision_score(y_test, yp, average='binary', zero_division=0):.4f}")
    print(f"    Recall   : {recall_score(y_test, yp, average='binary', zero_division=0):.4f}")


Detailed Metrics — Top 3:

  Logistic Regression:
    Accuracy : 0.7219
    F1       : 0.5137
    Precision: 0.6620
    Recall   : 0.4196

  FLAML:
    Accuracy : 0.7081
    F1       : 0.4918
    Precision: 0.6295
    Recall   : 0.4036

  CatBoost:
    Accuracy : 0.7125
    F1       : 0.4900
    Precision: 0.6462
    Recall   : 0.3946


## 24 · Error Analysis

Examine misclassifications from the best model.

In [16]:
best_name = top3_names[0]
best_pred = results[best_name]

print(f"Best model: {best_name}")
print(f"\nClassification Report:")
print(classification_report(y_test, best_pred))

errors = X_test.copy()
errors["actual"] = y_test.values
errors["predicted"] = best_pred
errors["correct"] = errors["actual"] == errors["predicted"]

n_errors = (~errors["correct"]).sum()
print(f"\nTotal errors: {n_errors} / {len(errors)} ({n_errors/len(errors)*100:.1f}%)")

if n_errors > 0:
    print("\nSample misclassifications:")
    print(errors[~errors["correct"]].head(10).to_string())

Best model: Logistic Regression

Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.88      0.81      1040
           1       0.66      0.42      0.51       560

    accuracy                           0.72      1600
   macro avg       0.70      0.65      0.66      1600
weighted avg       0.71      0.72      0.70      1600


Total errors: 445 / 1600 (27.8%)

Sample misclassifications:
      page_views  time_on_site_min  emails_opened  demos_attended  lead_source  company_size  engagement_score  actual  predicted  correct
2350           3               5.6              4               1          3.0           1.0                 9       1          0    False
1311           4               6.6              3               3          3.0           0.0                13       0          1    False
5649           6               0.1              3               0          1.0           2.0                 9       1          0    False

## 25 · Interpretation and Business Insight

**Key findings:**
- **Demo attendance** is the strongest conversion predictor.
- **Email engagement** and **page views** add complementary signal.
- **Enterprise** companies convert at higher rates.
- **Referral** leads outperform other sources.

**Business takeaway:** Prioritize demo scheduling and nurture referral channels.

## 26 · Limitations

1. Synthetic data with simplified engagement patterns.
2. No deal size or revenue potential.
3. Missing sales rep interaction data.
4. No temporal patterns (sales cycle length).
5. Lead source categories are too broad.

## 27 · How to Improve This Project

1. Use real CRM data (Salesforce, HubSpot exports).
2. Add sales cycle stage progression.
3. Include deal value for revenue-weighted scoring.
4. Add rep-level features (experience, win rate).
5. Build time-to-conversion survival model.

## 28 · Production Considerations

- Integrate with CRM via API for real-time scoring.
- Set threshold based on rep capacity.
- Monitor conversion rate drift monthly.
- Update scores as new engagement data arrives.

## 29 · Common Mistakes

1. Not accounting for lead source bias.
2. Including post-conversion data (leakage).
3. Using a single threshold for all segments.
4. Ignoring the cost of false negatives (missed opportunities).
5. Not validating on time-based splits.

## 30 · Mini Challenge / Exercises

1. Remove `demos_attended` — how much does F1 drop?
2. Create a `total_touches` feature (page_views + emails_opened) and compare.
3. Segment by company_size and train separate models.
4. Plot calibration curve for the best model.
5. Try different conversion thresholds (30%, 50%, 70%).

## 31 · Final Summary / Key Takeaways

1. **Demo attendance** is the strongest conversion signal.
2. **Multi-channel engagement** improves prediction.
3. **Enterprise leads** convert more reliably.
4. **Lead scoring** should output probabilities for prioritization.
5. **Referral channels** deliver the highest conversion rates.
6. **Always compare against a baseline** and validate on holdout data.

## Save Metrics

In [17]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "accuracy": round(float(accuracy_score(y_test, yp)), 4),
        "f1": round(float(f1_score(y_test, yp, average="binary")), 4),
        "precision": round(float(precision_score(y_test, yp, average="binary", zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, yp, average="binary", zero_division=0)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))

Metrics saved to E:\Github\Machine-Learning-Projects\Classification\Sales Lead Conversion Prediction\metrics.json
{
  "CatBoost": {
    "accuracy": 0.7125,
    "f1": 0.49,
    "precision": 0.6462,
    "recall": 0.3946
  },
  "LightGBM": {
    "accuracy": 0.7025,
    "f1": 0.4628,
    "precision": 0.6288,
    "recall": 0.3661
  },
  "Logistic Regression": {
    "accuracy": 0.7219,
    "f1": 0.5137,
    "precision": 0.662,
    "recall": 0.4196
  },
  "FLAML": {
    "accuracy": 0.7081,
    "f1": 0.4918,
    "precision": 0.6295,
    "recall": 0.4036
  }
}
